# STTran (ICCV 2021) on Action Genome — Colab (A100)

Goal: run **official STTran pretrained checkpoints** on Action Genome and start analysis.

This notebook:
- Mounts Google Drive
- Clones your fork
- Compiles extensions
- Downloads **official** weights from the upstream README Google Drive links:
  - `faster_rcnn_ag.pth` (detector)
  - `object_bbox_and_relationship_filtersmall.pkl`
  - `sttran_predcls.tar`, `sttran_sgcls.tar`, `sttran_sgdet.tar`
- Runs `test.py` using the chosen checkpoint

Prerequisite on Drive:
```
/content/drive/MyDrive/action_genome/
  annotations/
  frames/
  videos/   (optional for STTran; required only if you plan to re-extract frames)
```


In [ ]:
from google.colab import drive
import os, subprocess
drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/TommasoAiello08/STTran.git'  # change if needed
REPO_ROOT = '/content/STTran'
AG_ROOT = '/content/drive/MyDrive/action_genome'
WEIGHTS_DIR = '/content/drive/MyDrive/sttran_weights'

!rm -rf $REPO_ROOT
!git clone --depth 1 $REPO_URL $REPO_ROOT
%cd $REPO_ROOT
print(subprocess.check_output(['git','log','-1','--oneline']).decode())

assert os.path.isdir(os.path.join(AG_ROOT, 'annotations')), 'Missing Action Genome annotations/'
assert os.path.isdir(os.path.join(AG_ROOT, 'frames')), 'Missing Action Genome frames/'
print('AG_ROOT OK:', AG_ROOT)

In [ ]:
# Install minimal deps (Colab already has torch/torchvision)
!pip -q install cython dill easydict h5py opencv-python pandas tqdm pyyaml gdown

# Compile cython helpers
%cd lib/draw_rectangles
!python setup.py build_ext --inplace
%cd ../fpn/box_intersections_cpu
!python setup.py build_ext --inplace
%cd ../../..

# Compile Faster R-CNN extension
%cd fasterRCNN/lib
!python setup.py build_ext --inplace
%cd ../..

print('Build done')

In [ ]:
# Download official STTran Action Genome weights to Drive and copy into repo.
!python scripts/download_sttran_ag_weights.py --out_dir $WEIGHTS_DIR --link_into_repo

## Run evaluation

Pick ONE of the three official checkpoints below.

- PredCls: `ckpts/sttran_predcls.tar`
- SGCls: `ckpts/sttran_sgcls.tar`
- SGDet: `ckpts/sttran_sgdet.tar`


In [ ]:
# Change MODE + MODEL_PATH as desired.
MODE = 'predcls'   # predcls | sgcls | sgdet
MODEL_PATH = {
  'predcls': 'ckpts/sttran_predcls.tar',
  'sgcls':   'ckpts/sttran_sgcls.tar',
  'sgdet':   'ckpts/sttran_sgdet.tar',
}[MODE]

!python test.py -m $MODE -datasize large -data_path $AG_ROOT -model_path $MODEL_PATH